In [ ]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import os

file_path = "C:\Projects\OMSCS\Lizard_Classification\Anole_classifier\predictions_summary_no_duplicates_summary.xlsx"
sheet_names = ["true_positives", "false_positives", "mis_classification", "missed_detections"]

# Step 1: Create summary statistics table (now includes Q1 and Q3)
stats_list = []
confidence_data = {}

for sheet in sheet_names:
    df = pd.read_excel(file_path, sheet_name=sheet)
    col = df["Prediction Confidence"].dropna()
    q1 = col.quantile(0.25)
    q3 = col.quantile(0.75)
    confidence_data[sheet] = col.tolist()
    stats_list.append({
        "Sheet": sheet,
        "Mean": round(col.mean(), 3),
        "Median": round(col.median(), 3),
        "Q1 (25th)": round(q1, 3),
        "Q3 (75th)": round(q3, 3),
        "IQR": round(q3 - q1, 3),
        "Min": round(col.min(), 3),
        "Max": round(col.max(), 3)
    })

summary_df = pd.DataFrame(stats_list)

# Step 2: Write summary table to Excel
with pd.ExcelWriter(file_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    summary_df.to_excel(writer, sheet_name="summary", startrow=0, startcol=0, index=False)

# Step 3: Create improved box plot
fig, ax = plt.subplots(figsize=(10, 7))
box_data = [confidence_data[name] for name in sheet_names]
labels = ["True Positives", "False Positives", "Misclassifications", "Missed Detections"]

bp = ax.boxplot(
    box_data,
    labels=labels,
    patch_artist=True,
    widths=0.6,
    medianprops=dict(color='white', linewidth=2.5),   # White median line
    whiskerprops=dict(color='black', linewidth=1.5),
    capprops=dict(color='black', linewidth=1.5),
    flierprops=dict(marker='o', markerfacecolor='gray', markersize=5, alpha=0.6)
)

# Darker, vibrant fill colors
colors = ['#27ae60', '#c0392b', '#d35400', '#2471a3']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)

# Add mean markers (diamonds) for extra clarity
means = [sum(d) / len(d) for d in box_data]
for i, mean in enumerate(means):
    ax.plot(i + 1, mean, 'D', color='yellow', markersize=7, markeredgecolor='black', 
            markeredgewidth=0.5, zorder=5)

# Customize plot
ax.set_ylabel("Prediction Confidence", fontsize=12)
ax.set_title("Distribution of Prediction Confidence by Category", fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(0.2, 1.05)

# Add legend for elements not in the default legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='D', color='w', markerfacecolor='yellow', 
           markeredgecolor='black', markersize=8, label='Mean'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', 
           markersize=6, label='Outliers')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()

# Save box plot as image
boxplot_path = "boxplot_temp.png"
plt.savefig(boxplot_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()

# Step 4: Insert box plot image into the summary sheet
wb = load_workbook(file_path)
ws = wb["summary"]
img = Image(boxplot_path)

# Calculate a scaling factor to fit nicely
img.width = 480
img.height = 340
img.anchor = "A8"
ws.add_image(img)

# Adjust column width for readability
ws.column_dimensions['A'].width = 22
for col in ['B', 'C', 'D', 'E', 'F']:
    ws.column_dimensions[col].width = 12

wb.save(file_path)
os.remove(boxplot_path)
print("Summary sheet with improved box plot added successfully!")

C:\Users\wenha\AppData\Local\Temp\ipykernel_26168\3809238313.py:40: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(


Summary sheet with improved box plot added successfully!
